In [2]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import warnings

warnings.filterwarnings("ignore")

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything
from xlearner_hill import get_xlearner

# Tắt bớt log của Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for X-Learner Conversion...")
train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion' # BÀI TOÁN CONVERSION
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
# Chú ý: Dùng .values để xóa Pandas format, y ép kiểu .astype(int) vì là nhãn nhị phân
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(int) 
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(int)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(int)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    # Không gian tham số cho LightGBM (sẽ được dùng chung cho các mô hình con trong X-Learner)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Cập nhật thông số cố định
        current_params = params.copy()
        current_params['random_state'] = SEED
        current_params['verbose'] = -1
        
        # Khởi tạo X-Learner từ file xlearner_hill.py (chỉ định 'conversion')
        x_learner_model = get_xlearner(task_type='conversion', params=current_params)
        
        # Huấn luyện mô hình X-Learner của EconML
        # LƯU Ý: XLearner cần Y, T, X
        x_learner_model.fit(Y=y_train, T=t_train, X=X_train)
        
        # Dự đoán Uplift (CATE) trên tập Val bằng hàm .effect()
        uplift_pred_val = x_learner_model.effect(X_val).flatten()
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    # Trả về AUQC trung bình
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (X-Learner Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="X_Learner_Robust_Conv", sampler=fixed_sampler)
# X-Learner huấn luyện khá nhiều mô hình con, thời gian chạy sẽ lâu hơn S-Learner một chút
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")
print("Best Params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ X-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['verbose'] = -1
    
    # Khởi tạo lại model với best params và seed hiện tại
    final_x_learner = get_xlearner(task_type='conversion', params=best_params_final)
    
    # Train
    final_x_learner.fit(Y=y_train, T=t_train, X=X_train)
    
    # Predict
    uplift_pred_test = final_x_learner.effect(X_test).flatten()
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình và độ lệch chuẩn
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH X-LEARNER CONVERSION ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "x_learner_conversion_robust_results.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

Loading data for X-Learner Conversion...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (X-Learner Robust Conversion)...


Best trial: 42. Best value: 0.663926: 100%|██████████| 50/50 [03:47<00:00,  4.54s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.6639
Best Params:
  n_estimators: 111
  learning_rate: 0.0010275139449276354
  max_depth: 12
  num_leaves: 92
  min_child_samples: 57
  subsample: 0.5618502051946728
  colsample_bytree: 0.5224435716764941

🚀 ĐÁNH GIÁ X-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)
Seed 412312: AUUC=0.511, AUQC=0.512, Lift=0.005, KRCC=0.033
Seed 42: AUUC=0.518, AUQC=0.518, Lift=0.006, KRCC=0.040
Seed 1874: AUUC=0.534, AUQC=0.536, Lift=0.006, KRCC=0.039
Seed 902745: AUUC=0.544, AUQC=0.545, Lift=0.006, KRCC=0.058
Seed 1: AUUC=0.514, AUQC=0.516, Lift=0.005, KRCC=-0.067

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH X-LEARNER CONVERSION (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.524 ± 0.014
Mean AUQC: 0.526 ± 0.014
Mean Lift: 0.006 ± 0.001
Mean KRCC: 0.021 ± 0.050

💾 Đã lưu kết quả chi tiết từng seed vào: x_learner_conversion_robust_results.csv
